# Projection Plots for QMC Point Sets

Visualize 2D projections of different quasi-random and pseudo-random point sets.
This demo shows the structure and uniformity of various QMC sequences by
examining all pairwise dimension projections.

Based on the Python QMCPy demo `plot_proj_function.ipynb`.

*Note: The Python version uses `qp.plot_proj()` for matplotlib scatter plots.
This Julia version provides numerical summaries and text-based visualizations
since Plots.jl may not be available in all environments.*

In [ ]:
using QMC
import QMC: Uniform
using Statistics
using Printf

## 2D Point Sets

Compare IID, Halton, Digital Net, and Lattice in 2D.

In [ ]:
let n = 128
    println("=== 2D Point Set Statistics (n=$n) ===\n")
    for (name, dd) in [
        ("IID",         IIDStdUniform(2; seed=7)),
        ("Halton",      Halton(2; seed=7)),
        ("DigitalNetB2", DigitalNetB2(2; seed=7)),
        ("Lattice",     Lattice(2; seed=7)),
    ]
        pts = gen_samples(dd, n)
        println("$name:")
        for dim in 1:2
            col = pts[:, dim]
            println("  Dim $dim: mean=$(round(mean(col), digits=4)), " *
                    "std=$(round(std(col), digits=4)), " *
                    "min=$(round(minimum(col), digits=4)), " *
                    "max=$(round(maximum(col), digits=4))")
        end
        # L2 star discrepancy proxy: max gap between sorted points
        for dim in 1:2
            s = sort(pts[:, dim])
            gaps = diff(s)
            println("  Dim $dim max gap: $(round(maximum(gaps), digits=4)), " *
                    "mean gap: $(round(mean(gaps), digits=4))")
        end
        println("  Correlation(d1,d2): $(round(cor(pts[:,1], pts[:,2]), digits=4))")
        println()
    end
end

## 4D Projections

For 4-dimensional sequences, examine all 6 pairwise 2D projections.

In [ ]:
let n = 128, d = 4
    for (name, dd) in [
        ("Halton",       Halton(d; seed=7)),
        ("DigitalNetB2", DigitalNetB2(d; seed=7)),
        ("Lattice",      Lattice(d; seed=7)),
    ]
        pts = gen_samples(dd, n)
        println("=== $name ($d D, $n points) ===")
        println("Pairwise correlations:")
        for i in 1:d
            row = [@sprintf("% .4f", cor(pts[:,i], pts[:,j])) for j in 1:d]
            println("  ", join(row, "  "))
        end
        println()
        println("Per-dimension statistics:")
        for dim in 1:d
            col = pts[:, dim]
            s = sort(col)
            max_gap = maximum(diff(s))
            @printf("  Dim %d: mean=%.4f, max_gap=%.4f\n", dim, mean(col), max_gap)
        end
        println()
    end
end

## True Measure Transforms

Apply true measures (Gaussian, DistributionsWrapper) to QMC point sets.

In [ ]:
# Gaussian transform of IID points
dd = IIDStdUniform(2; seed=7)
tm = Gaussian(dd; mean=[2.0, 4.0], covariance=[9.0 4.0; 4.0 5.0])
n = 256
x_uniform = gen_samples(dd, n)
x_transformed = zeros(n, 2)
for i in 1:n
    x_transformed[i, :] = vec(transform(tm, x_uniform[i:i, :]))
end

println("Gaussian(mean=[2,4], cov=[[9,4],[4,5]]) transform:")
println("  Dim 1: mean=$(round(mean(x_transformed[:,1]), digits=2)), " *
        "std=$(round(std(x_transformed[:,1]), digits=2)) (target: mean=2, std=3)")
println("  Dim 2: mean=$(round(mean(x_transformed[:,2]), digits=2)), " *
        "std=$(round(std(x_transformed[:,2]), digits=2)) (target: mean=4, std≈2.24)")
println("  Correlation: $(round(cor(x_transformed[:,1], x_transformed[:,2]), digits=3)) " *
        "(target: $(round(4/sqrt(9*5), digits=3)))")

## Text-Based Scatter Plot

A simple ASCII scatter plot to visualize point distributions.

In [ ]:
function ascii_scatter(x, y; width=60, height=20, title="")
    xmin, xmax = minimum(x), maximum(x)
    ymin, ymax = minimum(y), maximum(y)
    xr = xmax - xmin; yr = ymax - ymin
    if xr == 0; xr = 1.0; end
    if yr == 0; yr = 1.0; end
    grid = fill(' ', height, width)
    for i in eachindex(x)
        c = clamp(round(Int, (x[i] - xmin) / xr * (width - 1)) + 1, 1, width)
        r = clamp(height - round(Int, (y[i] - ymin) / yr * (height - 1)), 1, height)
        grid[r, c] = '·'
    end
    !isempty(title) && println(title)
    for r in 1:height
        println(String(grid[r, :]))
    end
    @printf("x: [%.2f, %.2f]  y: [%.2f, %.2f]\n", xmin, xmax, ymin, ymax)
end

# Compare IID vs Lattice
n = 200
dd_iid = IIDStdUniform(2; seed=42)
dd_lat = Lattice(2; seed=42)
x_iid = gen_samples(dd_iid, n)
x_lat = gen_samples(dd_lat, n)

ascii_scatter(x_iid[:,1], x_iid[:,2]; title="\nIID ($n points):")
println()
ascii_scatter(x_lat[:,1], x_lat[:,2]; title="Lattice ($n points):")